# The Price is Right

## 第八周课程安排

第一天：Modal.com 和 SpecialistAgent  
第二天：RAG、FrontierAgent、Ensemble Agent  
第三天：ScannerAgent、MessengerAgent   
第四天：AutonomousPlannerAgent  
第五天：The Price Is Right 终章


现在是时候介绍 Planning Agent 了

In [1]:
import json
from openai import OpenAI
from dotenv import load_dotenv
from agents.scanner_agent import ScannerAgent
import chromadb
import logging
load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-5.1"

## 从一些测试数据开始

In [2]:
test_results = ScannerAgent().test_scan()
test_results

DealSelection(deals=[Deal(product_description="The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.", price=178.0, url='https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142'), Deal(product_description='The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and

## 现在让我们创建 3 个模拟函数..

In [3]:
def scan_the_internet_for_bargains() -> str:
    """ 此工具扫描互联网上的优惠信息，获取精选的有前景优惠列表 """
    print("模拟扫描互联网的函数 - 返回硬编码的优惠集合")
    return test_results.model_dump_json()

In [4]:
def estimate_true_value(description: str) -> str:
    """
    此工具根据产品的文本描述估算其真实价值
    """
    print(f"模拟估算真实价值的函数，描述：{description[:20]}... - 总是返回 $300")
    return f"Product {description} has an estimated true value of $300"

In [5]:
def notify_user_of_deal(description: str, deal_price: float, estimated_true_value: float, url: str) -> str:
    """
    此工具通知用户一个优惠信息，包括描述、优惠价格和估算的真实价值
    """
    print(f"模拟通知用户的函数：{description}，价格：{deal_price}，估算价值：{estimated_true_value}")
    return "notification sent ok"

### 让我们试试这些函数

In [6]:
notify_user_of_deal("a new iphone", 100, 1000, "https://www.apple.com/iphone")

模拟通知用户的函数：a new iphone，价格：100，估算价值：1000


'notification sent ok'

### 现在是一大块 JSON

In [7]:
scan_function = {
        "name": "scan_the_internet_for_bargains",
        "description": "Returns top bargains scraped from the internet along with the price each item is being offered for",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        }
    }

estimate_function = {
    "name": "estimate_true_value",
    "description": "Given the description of an item, estimate how much it is actually worth",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item to be estimated"
            },
        },
        "required": ["description"],
        "additionalProperties": False
    }
}

notify_function = {
    "name": "notify_user_of_deal",
    "description": "Send the user a push notification about the single most compelling deal; only call this one time",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item itself scraped from the internet"
            },
            "deal_price": {
                "type": "number",
                "description": "The price offered by this deal scraped from the internet"
            }
            ,
            "estimated_true_value": {
                "type": "number",
                "description": "The estimated actual value that this is worth"
            }
            ,
            "url": {
                "type": "string",
                "description": "The URL of this deal as scraped from the internet"
            }
        },
        "required": ["description", "deal_price", "estimated_true_value", "url"],
        "additionalProperties": False
    }
}

In [8]:
tools = [{"type": "function", "function": scan_function},
 {"type": "function", "function": estimate_function},
 {"type": "function", "function": notify_function}
 ]

In [9]:
tools

[{'type': 'function',
  'function': {'name': 'scan_the_internet_for_bargains',
   'description': 'Returns top bargains scraped from the internet along with the price each item is being offered for',
   'parameters': {'type': 'object',
    'properties': {},
    'required': [],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'estimate_true_value',
   'description': 'Given the description of an item, estimate how much it is actually worth',
   'parameters': {'type': 'object',
    'properties': {'description': {'type': 'string',
      'description': 'The description of the item to be estimated'}},
    'required': ['description'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'notify_user_of_deal',
   'description': 'Send the user a push notification about the single most compelling deal; only call this one time',
   'parameters': {'type': 'object',
    'properties': {'description': {'type': 'string',
      'description

In [11]:
def handle_tool_call(message):
    """
    实际调用与此消息关联的工具
    """
    results = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
    ##  if tool_name == scan_the_internet_for_bargains:
    ##    scan_the_internet_for_bargains()
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [12]:
system_message = "You find great deals on bargain products using your tools, and notify the user of the best bargain."
user_message = """
First, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.
Then pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.
Then just reply OK to indicate success.
"""
messages = [{"role": "system", "content": system_message},{"role": "user", "content": user_message}]

In [13]:
messages

[{'role': 'system',
  'content': 'You find great deals on bargain products using your tools, and notify the user of the best bargain.'},
 {'role': 'user',
  'content': '\nFirst, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.\nThen pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.\nThen just reply OK to indicate success.\n'}]

In [14]:
done = False
while not done:
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        results = handle_tool_call(message)
        messages.append(message)
        messages.extend(results)
    else:
        done = True
response.choices[0].message.content

模拟扫描互联网的函数 - 返回硬编码的优惠集合
模拟估算真实价值的函数，描述：The Hisense R6 Serie... - 总是返回 $300
模拟估算真实价值的函数，描述：The Poly Studio P21 ... - 总是返回 $300
模拟估算真实价值的函数，描述：The Lenovo IdeaPad S... - 总是返回 $300
模拟估算真实价值的函数，描述：The Dell G15 gaming ... - 总是返回 $300
模拟通知用户的函数：The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom control, along with an ambient light sensor to adjust the vanity lighting as needed. It also supports 5W wireless charging for mobile devices, making it an all-in-one solution for home offices.，价格：30，估算价值：300


'OK'

## 现在.. 进入自主规划代理

将模拟函数替换为真实函数！

In [15]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [16]:
DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection('products')

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


In [17]:
from agents.autonomous_planning_agent import AutonomousPlanningAgent
agent = AutonomousPlanningAgent(collection)

INFO:datasets:PyTorch version 2.6.0 available.
16:04:18 - LiteLLM:WARNING: get_model_cost_map.py:213 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016). Falling back to local backup.
INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-

In [21]:
agent.plan()

INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent is kicking off a run
INFO:openai._base_client:Retrying request to /chat/completions in 0.443493 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 0.830349 seconds
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is calling scanner
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI
INFO:openai._base_client:Retrying request to /chat/completions in 0.445832 seconds
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
16:09:43 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama3.2;

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $16.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $36.26
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $19.72
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
16:10:31 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama3.2; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= llama3.2; provider = ollama
16:10:32 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Age

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $1249.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $368.65
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $1086.06
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
16:10:34 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama3.2; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= llama3.2; provider = ollama
16:10:35 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensembl

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $160.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $311.79
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $194.18
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
16:10:39 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama3.2; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= llama3.2; provider = ollama
16:10:40 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $1599.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $191.44
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $1398.24
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is notifying user
INFO:root:[Messaging Agent] Messaging Agent is using GPT-5.1 to craft the message
16:10:44 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= gpt-5.1; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5.1; provider = openai
16:10:46 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Me

Opportunity(deal=Deal(product_description='The Jaszdot 32" Floor Standing Digital Signage Display is a movable kiosk-style screen intended for retail spaces, lobbies, and events. It features a 32-inch 1080p display with 178-degree wide viewing angles so content can be seen clearly from various positions. A 20-point capacitive touchscreen supports multi-touch gestures for interactive presentations and self-service applications, while intelligent split-screen controls enable simultaneous display of different media formats. The unit includes an adjustable stand and silent caster wheels for easy repositioning, along with remote content management via an Android mobile app.', price=450.0, url='https://www.dealnews.com/Jaszdot-32-Floor-Standing-Digital-Signage-Display-for-450-free-shipping/22184911.html?iref=rss-c142'), estimate=1086.0648254394532, discount=636.0648254394532)